In [ ]:
%pip install transformers
%pip install accelerate
%pip install emoji wordsegment
%pip install huggingface_hub
%pip install torch --index-url https://download.pytorch.org/whl/cu124
%pip install pandas scikit-learn
%pip install ftfy regex tqdm
%pip install PyEvALL
%pip install torchaudio==2.6.0+cu124 --index-url https://download.pytorch.org/whl/cu124
%pip install -U openai-whisper
%pip install peft

In [ ]:
LABELS_2_1 = ['YES', 'NO']
LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT']
LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
               'SEXUAL-VIOLENCE', 'OBJECTIFICATION', 'STEREOTYPING-DOMINANCE']

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import os
import torch.nn as nn
import torch.nn.functional as F
import emoji
from wordsegment import segment
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.preprocessing import RobustScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    TrainingArguments,
    Trainer,
    XLMRobertaTokenizer,
    XLMRobertaModel,
    VideoMAEModel,
    EarlyStoppingCallback,
    CLIPProcessor,
    CLIPVisionModel
)
import random
from torch.optim.swa_utils import AveragedModel, SWALR
import json
import os
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
import whisper
import torchaudio
from pathlib import Path
from tqdm import tqdm
import tempfile
from peft import LoraConfig, get_peft_model
from transformers import set_seed

In [ ]:
def set_all_seeds(seed: int):
    """Set every RNG that affects training reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

In [ ]:
SEED = 42   # change this for each run
set_all_seeds(SEED)

In [ ]:


# Viewer ID → index mapping (fixed panel of 10)
VIEWER_TO_IDX = {
    "EN1_2": 0, "EN2_2": 1, "EN3_2": 2, "EN4_2": 3, "EN5_2": 4,
    "ES1_2": 5, "ES2_2": 6, "ES3_2": 7, "ES4_2": 8, "ES5_2": 9,
}

# RT imputation values per language (from Day 1 analysis)
RT_IMPUTE = {"en": 29.0, "es": 17.5}
RT_CLIP   = 120.0
THRESHOLD_GRID = np.arange(0.05, 0.65, 0.05).tolist()

In [ ]:
def load_frames(frames_dir: str, item_id: str) -> torch.Tensor:
    """
    Load pre-extracted video frames tensor.
    Returns [8, 3, 224, 224] float tensor.
    Falls back to zeros if file missing.
    """
    path = Path(frames_dir) / f"{item_id}.pt"
    try:
        frames = torch.load(path, weights_only=True)
        return frames.float()
    except Exception:
        print(f"[WARN] Frames not found for {item_id}, using zeros")
        return torch.zeros(16, 3, 224, 224, dtype=torch.float32)


In [ ]:
def load_audio_features(audio_dir: str, item_id: str) -> torch.Tensor:
    """
    Load pre-extracted Whisper encoder features.
    Returns [1280] float tensor.
    Falls back to zeros if file missing.
    """
    path = Path(audio_dir) / f"{item_id}.pt"
    try:
        features = torch.load(path, weights_only=True)
        return features.float()
    except Exception:
        print(f"[WARN] Audio features not found for {item_id}, using zeros")
        return torch.zeros(1280, dtype=torch.float32)


In [ ]:
def safe_float(val, default=0.0):
    """Convert to float safely — handles None, NaN, non-numeric."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return default

In [ ]:
def build_bio_vector(
    sensorial: dict,
    lang: str,
    scaler,                   # fitted RobustScaler from training set
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Extract and preprocess biosignal features from sensorial dict.

    Returns:
        bio_vec:  [16] averaged ET+HR across 2 users (robust scaled)
        cvpc:     [1]  cosine similarity between the 2 users' raw vectors
        rt_norm:  [1]  mean reaction time (clipped, scaled)
        fix_norm: [1]  mean fixation count (scaled)
        user_ids: [2]  viewer indices into the embedding table
    """
    users     = sensorial["users"]                    # e.g. ["ES1_2", "ES3_2"]
    et_data   = sensorial["modalities"]["ET"]["by_user"]
    hr_data   = sensorial["modalities"]["HR"]["by_user"]

    user_vecs = []
    user_ids  = []
    rt_vals   = []
    fix_vals  = []

    for user in users:
        et = et_data[user]
        hr = hr_data[user]

        # Reaction time: clip outliers + impute NaN
        rt = et.get("reaction_time", None)
        if rt is None or safe_float(rt) != safe_float(rt):  # NaN check
            rt = RT_IMPUTE[lang]
        rt = min(safe_float(rt), RT_CLIP)
        rt_vals.append(rt)
        
        fix = safe_float(et.get("fixations_count"), 0.0)
        fix_vals.append(fix)
        
        vec = np.array([
            safe_float(et.get("3d_eye_states_pupil diameter left [mm]_mean")),
            safe_float(et.get("3d_eye_states_pupil diameter left [mm]_std")),
            safe_float(et.get("3d_eye_states_pupil diameter right [mm]_mean")),
            safe_float(et.get("3d_eye_states_pupil diameter right [mm]_std")),
            safe_float(et.get("blinks_count")),
            safe_float(et.get("blinks_duration_mean_ns")),
            safe_float(et.get("fixations_count")),
            safe_float(et.get("fixations_duration_mean_ns")),
            safe_float(et.get("fixations_duration_std_ns")),
            safe_float(et.get("saccades_count")),
            safe_float(et.get("saccades_duration_mean_ns")),
            rt,
            safe_float(hr.get("garmin_hr_mean")),
            safe_float(hr.get("garmin_hr_std")),
            safe_float(hr.get("garmin_hr_max")),
            safe_float(hr.get("garmin_hr_min")),
        ], dtype=np.float32)

        user_vecs.append(vec)
        user_ids.append(VIEWER_TO_IDX[user])

    # CVPC: cosine similarity between the 2 raw vectors
    v1, v2     = user_vecs[0], user_vecs[1]
    v1_scaled  = scaler.transform(v1.reshape(1, -1)).flatten()   # [16]
    v2_scaled  = scaler.transform(v2.reshape(1, -1)).flatten()   # [16]

    # CVPC: cosine similarity on standardised vectors
    dot   = np.dot(v1_scaled, v2_scaled)
    norms = (np.linalg.norm(v1_scaled) * np.linalg.norm(v2_scaled)) + 1e-8
    cvpc  = float(dot / norms)

    # Average across the 2 standardised vectors → [16]
    bio_scaled = np.mean([v1_scaled, v2_scaled], axis=0)

    # Scalar features for τ-network (also scaled via scaler internals)
    rt_mean  = float(np.mean(rt_vals))
    fix_mean = float(np.mean(fix_vals))

    return (
        torch.tensor(bio_scaled,           dtype=torch.float32),  # [16]
        torch.tensor([cvpc],               dtype=torch.float32),  # [1]
        torch.tensor([rt_mean],            dtype=torch.float32),  # [1]
        torch.tensor([fix_mean],           dtype=torch.float32),  # [1]
        torch.tensor(user_ids,             dtype=torch.long),     # [2]
    )


In [ ]:
def build_soft_label_a(labels: list[str]) -> torch.Tensor:
    """
    Build soft label for Task A (binary sexism).
    Proportion of YES votes. UNKNOWN votes are excluded.
    Returns -100.0 sentinel if all labels are UNKNOWN.
    """
    valid = [l for l in labels if l != "UNKNOWN"]
    if len(valid) == 0:
        return torch.tensor(-100.0, dtype=torch.float32)
    soft = sum(1 for l in valid if l == "YES") / len(valid)
    return torch.tensor(soft, dtype=torch.float32)


In [ ]:
def build_soft_label_b(labels: list[str]) -> torch.Tensor:
    """
    Build soft label for Task B (NONE / JUDGEMENTAL / DIRECT).
    Order matches LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT'].
    '-' is remapped to 'NONE' (not sexist). UNKNOWN votes are excluded.
    Returns [-1.0, -1.0, -1.0] sentinel if all labels are UNKNOWN.
    """
    remapped = []
    for l in labels:
        if l == "UNKNOWN":
            continue
        remapped.append("NONE" if l == "-" else l)

    if len(remapped) == 0:
        return torch.full((len(LABELS_2_2),), -1.0, dtype=torch.float32)

    counts = {lbl: 0 for lbl in LABELS_2_2}
    for l in remapped:
        if l in counts:
            counts[l] += 1

    soft = np.array([counts[lbl] / len(remapped) for lbl in LABELS_2_2], dtype=np.float32)
    return torch.tensor(soft, dtype=torch.float32)

In [ ]:
def build_soft_label_c(labels: list[list[str]]) -> torch.Tensor:
    """
    Build soft label for Task C (6 multi-label categories).
    Order matches LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', ...].
    Each annotator can assign multiple categories; '-' is remapped to 'NONE'.
    Returns [-1.0, ...] sentinel if all labels are UNKNOWN or ['-'].
    """
    remapped = []
    for ann_labels in labels:
        cleaned = []
        for lbl in ann_labels:
            if lbl == "UNKNOWN":
                continue
            cleaned.append("NONE" if lbl == "-" else lbl)
        if cleaned:
            remapped.append(cleaned)

    if len(remapped) == 0:
        return torch.full((len(LABELS_2_3),), -1.0, dtype=torch.float32)

    counts = {lbl: 0 for lbl in LABELS_2_3}
    for ann_labels in remapped:
        for lbl in ann_labels:
            if lbl in counts:
                counts[lbl] += 1

    soft = np.array([counts[lbl] / len(remapped) for lbl in LABELS_2_3], dtype=np.float32)
    return torch.tensor(soft, dtype=torch.float32)

In [ ]:
def tokenize_text(text: str, tokenizer, max_length: int) -> dict:
    """Tokenize transcript text with padding and truncation."""
    return tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

In [ ]:
def collect_bio_vectors(dataframe: pd.DataFrame) -> np.ndarray:
    vectors = []
    for _, row in dataframe.iterrows():
        sensorial = row["sensorial"]
        lang      = row["lang"]
        et_data   = sensorial["modalities"]["ET"]["by_user"]
        hr_data   = sensorial["modalities"]["HR"]["by_user"]
        for user in sensorial["users"]:
            et = et_data[user]
            hr = hr_data[user]
            rt = safe_float(et.get("reaction_time"), default=float("nan"))
            if rt != rt:  # NaN check
                rt = RT_IMPUTE[lang]
            rt = min(rt, RT_CLIP)
            vec = np.array([
                safe_float(et.get("3d_eye_states_pupil diameter left [mm]_mean")),
                safe_float(et.get("3d_eye_states_pupil diameter left [mm]_std")),
                safe_float(et.get("3d_eye_states_pupil diameter right [mm]_mean")),
                safe_float(et.get("3d_eye_states_pupil diameter right [mm]_std")),
                safe_float(et.get("blinks_count")),
                safe_float(et.get("blinks_duration_mean_ns")),
                safe_float(et.get("fixations_count")),
                safe_float(et.get("fixations_duration_mean_ns")),
                safe_float(et.get("fixations_duration_std_ns")),
                safe_float(et.get("saccades_count")),
                safe_float(et.get("saccades_duration_mean_ns")),
                rt,
                safe_float(hr.get("garmin_hr_mean")),
                safe_float(hr.get("garmin_hr_std")),
                safe_float(hr.get("garmin_hr_max")),
                safe_float(hr.get("garmin_hr_min")),
            ], dtype=np.float32)
            vectors.append(vec)
    return np.stack(vectors)

In [ ]:
class EXISTVideoDataset(Dataset):
    def __init__(
        self,
        dataframe,            # pd.DataFrame instead of dict
        tokenizer,
        scaler,
        frames_dir: str,
        audio_dir: str,
        max_length: int = 128,
    ):
        self.tokenizer  = tokenizer
        self.scaler     = scaler
        self.frames_dir = frames_dir
        self.audio_dir  = audio_dir
        self.max_length = max_length

        # Store columns as arrays for fast indexing (same pattern as your original)
        self.item_ids   = dataframe["id_EXIST"].values
        self.langs      = dataframe["lang"].values
        self.texts      = dataframe["text"].values
        self.sensorials = dataframe["sensorial"].values    # list of dicts
        self.labels_a   = dataframe["labels_task3_1"].values
        self.labels_b   = dataframe["labels_task3_2"].values
        self.labels_c   = dataframe["labels_task3_3"].values

    def __len__(self) -> int:
        return len(self.item_ids)

    def __getitem__(self, idx: int) -> dict:
            item_id   = self.item_ids[idx]
            lang      = self.langs[idx]
            text      = self.texts[idx]
            sensorial = self.sensorials[idx]
    
            frames         = load_frames(self.frames_dir, item_id)
            audio_features = load_audio_features(self.audio_dir, item_id)
            tokenized      = tokenize_text(text, self.tokenizer, self.max_length)
    
            bio_vec, cvpc, rt_norm, fix_norm, user_ids = build_bio_vector(
                sensorial, lang, self.scaler
            )
    
            label_a = build_soft_label_a(self.labels_a[idx])
            label_b = build_soft_label_b(self.labels_b[idx])
            label_c = build_soft_label_c(self.labels_c[idx])
    
            return {
                "frames":         frames,
                "audio":          audio_features,
                "input_ids":      tokenized["input_ids"].squeeze(0),
                "attention_mask": tokenized["attention_mask"].squeeze(0),
                "bio_vec":        bio_vec,
                "cvpc":           cvpc,
                "rt_norm":        rt_norm,
                "fix_norm":       fix_norm,
                "user_ids":       user_ids,
                "labels_a":       label_a,
                "labels_b":       label_b,
                "labels_c":       label_c,
            }

In [ ]:
train_df = pd.read_json('./Dataset/training/EXIST2026_training.json').transpose()
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["labels_task3_1"].apply(
        lambda x: "YES" if "YES" in x else "NO"   # stratify by sexist/not
    ),
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

In [ ]:
train_bio_matrix = collect_bio_vectors(train_df)
scaler = RobustScaler().fit(train_bio_matrix)

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")

train_dataset = EXISTVideoDataset(train_df, tokenizer, scaler,
                                  "frames/train/", "audio_features/train/")
val_dataset   = EXISTVideoDataset(val_df,   tokenizer, scaler,
                                  "frames/train/", "audio_features/train/")

In [ ]:
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:20s}: {v.shape if hasattr(v, 'shape') else v}")

# VideoMae with LoRA config

In [ ]:
videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value"],  # attention Q and V only
)
videomae = get_peft_model(videomae, lora_cfg)
videomae.print_trainable_parameters()

# Roberta LoRa config

In [ ]:
xlmr = XLMRobertaModel.from_pretrained("xlm-roberta-large")

lora_cfg_text = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value"],
)
xlmr = get_peft_model(xlmr, lora_cfg_text)
xlmr.print_trainable_parameters()

# Model class creation

In [ ]:
from transformers import TrainerCallback
from torch.optim.swa_utils import AveragedModel, update_bn

class SWACallback(TrainerCallback):
    """Stochastic Weight Averaging callback for HuggingFace Trainer.

    Maintains a running average of model weights starting from `swa_start_epoch`.
    At the end of training, swaps the trainer's model with the averaged one and
    updates BatchNorm statistics if any BN layers exist.
    """
    def __init__(self, swa_start_epoch: int = 8, swa_lr: float = None):
        self.swa_start_epoch = swa_start_epoch
        self.swa_lr = swa_lr
        self.swa_model = None
        self.swa_n = 0   # number of models averaged so far

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        # Build the averaged model wrapper. It lives on CPU initially;
        # we'll move it once it sees its first update.
        self.swa_model = AveragedModel(model)
        # Move to same device as the model
        device = next(model.parameters()).device
        self.swa_model = self.swa_model.to(device)
        print(f"[SWA] Initialised. Averaging starts at epoch {self.swa_start_epoch}.")

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if state.epoch >= self.swa_start_epoch:
            self.swa_model.update_parameters(model)
            self.swa_n += 1
            print(f"[SWA] Epoch {epoch}: averaged model updated (n={self.swa_n})")

    def on_train_end(self, args, state, control, model=None, train_dataloader=None, **kwargs):
        if self.swa_n == 0:
            print("[SWA] No epochs averaged — SWA disabled in final model.")
            return

        print(f"[SWA] Training complete. Final SWA model averaged over {self.swa_n} epochs.")
        # Update BN statistics on training data (no-op if no BN layers exist)
        # Your model has no BN currently, so this is a safe no-op.
        # If you add BN later, uncomment:
        # update_bn(train_dataloader, self.swa_model, device=next(model.parameters()).device)

        # Copy SWA weights into the trainer's model so eval/save uses them.
        # AveragedModel wraps the original — unwrap with .module
        model.load_state_dict(self.swa_model.module.state_dict())
        print("[SWA] Averaged weights loaded into model. Final eval will use SWA model.")

In [ ]:
class EXISTVideoModel(nn.Module):
    def __init__(self, videomae, xlmr):
        super().__init__()

        self.videomae    = videomae
        self.xlmr        = xlmr

        self.audio_proj  = nn.Linear(1280, 768)

        self.viewer_emb  = nn.Embedding(10, 32)
        self.viewer_proj = nn.Linear(32, 16)
        self.bio_mlp     = nn.Sequential(
            nn.Linear(17, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
        )

        self.fusion = nn.Sequential(
            nn.Linear(768 + 768 + 1024 + 32, 1024),   # video + audio + TEXT_LARGE + bio
            nn.GELU(),
            nn.Dropout(0.1),
        )

        self.head_a = nn.Linear(1024, 1)
        self.head_b = nn.Linear(1024, 3)
        self.head_c = nn.Linear(1024, 6)
    def gradient_checkpointing_enable(self, **kwargs):
        self.videomae.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.xlmr.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    def forward(
            self,
            frames,
            audio,
            input_ids,
            attention_mask,
            bio_vec,
            cvpc,
            rt_norm,
            fix_norm,
            user_ids,
            labels_a=None,
            labels_b=None,
            labels_c=None,
            # ── Ablation flags ────────────────────────────────────────────
            use_bio=True,          # False → zero out all bio signal
            use_viewer_emb=True,   # False → skip per-viewer baseline subtraction
            modality_dropout_prob=0.15,  # probability of dropping ONE modality per step (training only)
            **kwargs,
        ):
            # ── Modality dropout (training only) ──────────────────────────
            # Drop ONE modality per step with probability modality_dropout_prob each.
            # Total drop probability = 4 * modality_dropout_prob (cap at 0.6).
            # Bio is included only when use_bio=True — never drop a modality that's
            # already disabled by an ablation flag.
            drop_video = drop_audio = drop_text = drop_bio = False
            if self.training and modality_dropout_prob > 0:
                r = torch.rand(1).item()
                p = modality_dropout_prob
                if   r < p:               drop_video = True
                elif r < 2 * p:           drop_audio = True
                elif r < 3 * p:           drop_text  = True
                elif r < 4 * p and use_bio:
                                          drop_bio   = True
                # else: keep all modalities
    
            # ── Video ─────────────────────────────────────────────────────
            if drop_video:
                vid_feat = torch.zeros(frames.shape[0], 768, device=frames.device,
                                       dtype=self.audio_proj.weight.dtype)
            else:
                vid_out  = self.videomae(pixel_values=frames).last_hidden_state  # [B, T, 768]
                vid_feat = vid_out.mean(dim=1)                                    # [B, 768]
    
            # ── Audio ─────────────────────────────────────────────────────
            if drop_audio:
                aud_feat = torch.zeros(frames.shape[0], 768, device=frames.device,
                                       dtype=self.audio_proj.weight.dtype)
            else:
                aud_feat = self.audio_proj(audio)                                # [B, 768]
    
            # ── Text ──────────────────────────────────────────────────────
            if drop_text:
                txt_feat = torch.zeros(frames.shape[0], 1024, device=frames.device,
                                       dtype=self.audio_proj.weight.dtype)
            else:
                txt_feat = self.xlmr(input_ids=input_ids,
                                     attention_mask=attention_mask).last_hidden_state[:, 0, :]  # [B, 1024]
    
            # ── Bio ───────────────────────────────────────────────────────
            if use_bio and not drop_bio:
                if use_viewer_emb:
                    # subtract learned per-viewer physiological baseline
                    base1    = self.viewer_proj(self.viewer_emb(user_ids[:, 0]))   # [B, 16]
                    base2    = self.viewer_proj(self.viewer_emb(user_ids[:, 1]))   # [B, 16]
                    bio_norm = bio_vec - (base1 + base2) / 2                       # [B, 16]
                else:
                    bio_norm = bio_vec                                              # [B, 16] raw
    
                bio_input = torch.cat([bio_norm, cvpc], dim=-1)                    # [B, 17]
                bio_feat  = self.bio_mlp(bio_input)                                # [B, 128]
            else:
                # B0 ablation OR modality dropout fired on bio
                bio_feat = torch.zeros(frames.shape[0], 32, device=frames.device,
                                       dtype=self.audio_proj.weight.dtype)         # [B, 128]
    
            # ── Fusion ────────────────────────────────────────────────────
            fused = torch.cat([vid_feat, aud_feat, txt_feat, bio_feat], dim=-1)    # [B, 2432]
            fused = self.fusion(fused)                                              # [B, 1024]
    
            # ── Task logits ───────────────────────────────────────────────
            logits_a = self.head_a(fused).squeeze(-1)   # [B]
            logits_b = self.head_b(fused)               # [B, 3]
            logits_c = self.head_c(fused)               # [B, 6]                                  
    
            loss = None
            if labels_a is not None:
                loss = compute_loss(logits_a, logits_b, logits_c,
                                    labels_a, labels_b, labels_c)
                if loss.dim() != 0:
                    loss = loss.mean()
            if loss is not None:
                return (loss, logits_a, logits_b, logits_c)
            else:
                return (logits_a, logits_b, logits_c)

# Loss function

In [ ]:
def loss_task_a(logits_a, labels_a):
    mask = (labels_a != -100.0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_a.device)   # skip empty batch
    loss = F.binary_cross_entropy_with_logits(logits_a, labels_a.clamp(0,1), reduction="none")
    return (loss * mask).sum() / mask.sum()

def loss_task_b(logits_b, labels_b):
    mask = (labels_b[:, 0] != -1.0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_b.device)
    log_pb = F.log_softmax(logits_b, dim=-1)
    loss   = -(labels_b.clamp(0,1) * log_pb).sum(dim=-1)
    return (loss * mask).sum() / mask.sum()

def loss_task_c(logits_c, labels_c):
    mask = (labels_c[:, 0] != -1.0).float()      # only sentinel-mask
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_c.device)
    loss = F.binary_cross_entropy_with_logits(
        logits_c, labels_c.clamp(0,1), reduction="none"
    ).mean(dim=-1)
    return (loss * mask).sum() / mask.sum()

def compute_loss(logits_a, logits_b, logits_c, labels_a, labels_b, labels_c):
    la = loss_task_a(logits_a, labels_a)
    lb = loss_task_b(logits_b, labels_b)
    lc = loss_task_c(logits_c, labels_c)
    return la + 0.7*lb + 0.7*lc

In [ ]:
model = EXISTVideoModel(videomae, xlmr)

In [ ]:
def get_optimizer(model):
    # LoRA params in pretrained encoders → lower LR (fine-tuning)
    lora_params = [p for n, p in model.named_parameters()
                   if "lora" in n and p.requires_grad]

    # viewer embeddings → no weight decay (embeddings collapse with decay)
    viewer_params = [p for n, p in model.named_parameters()
                     if "viewer_emb" in n and p.requires_grad]

    # everything else (fusion, heads, bio_mlp, tau_mlp, audio_proj) → higher LR
    new_params = [p for n, p in model.named_parameters()
                  if "lora" not in n and "viewer_emb" not in n and p.requires_grad]

    optimizer = torch.optim.AdamW([
        {"params": lora_params,   "lr": 1e-4, "weight_decay": 0.01},
        {"params": viewer_params, "lr": 1e-3, "weight_decay": 0.0},   # no decay
        {"params": new_params,    "lr": 1e-3, "weight_decay": 0.05},
    ])
    return optimizer

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./checkpoints_seed_{SEED}",
    num_train_epochs=15,

    # batch: effective size = 4 * 8 = 32
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,

    # memory
    bf16=True, 
    fp16=False,
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    # eval + saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_task_a_icmsoft_soft",
    greater_is_better=True,
    use_cache=False,
    # LR schedule
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    # critical — dataset has custom fields Trainer doesn't know about
    remove_unused_columns=False,

    report_to="none",
    logging_steps=10,
    seed=SEED,
    data_seed=SEED,
)

In [ ]:
def _search_thresholds_task_c(probs_c, targets_c, ids_c, temp_dir):
    """Greedy coordinate search over per-class thresholds for Task C hard ICMNorm.
    
    Optimizes one class at a time holding the others fixed.
    12 candidates × 6 classes = 72 PyEvALL calls per epoch.

    Returns: (best_thresholds: list[float], final_scores: dict)
    """
    best_thresholds = [0.5] * len(LABELS_2_3)

    for class_idx in range(len(LABELS_2_3)):
        best_t   = best_thresholds[class_idx]
        best_icm = -np.inf

        for t in THRESHOLD_GRID:
            candidate = best_thresholds.copy()
            candidate[class_idx] = float(t)

            scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                  temp_dir=temp_dir, thresholds_c=candidate)
            icm = scores.get('ICMNorm', -np.inf)
            if icm > best_icm:
                best_icm = icm
                best_t   = float(t)

        best_thresholds[class_idx] = best_t

    # Final eval with all thresholds locked in
    final_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                temp_dir=temp_dir, thresholds_c=best_thresholds)
    return best_thresholds, final_scores


In [ ]:
def _format_for_pyevall(probs, sample_ids, task, evaluation, thresholds_c=None):
    """Convert probability arrays to PyEvALL JSON format.
    
    thresholds_c: optional list of 6 floats for Task C hard mode.
                  If None, defaults to 0.5 for all classes.
    """
    out = []
    for i, sid in enumerate(sample_ids):
        entry = {"test_case": "EXIST2025", "id": str(sid)}

        if task == '2_1':
            p_no  = float(probs[i][0])
            p_yes = float(probs[i][1])
            if evaluation == 'hard':
                entry["value"] = "YES" if p_yes >= p_no else "NO"
            else:
                entry["value"] = {"YES": p_yes, "NO": p_no}

        elif task == '2_2':
            label_map = {"NONE": "NO", "JUDGEMENTAL": "JUDGEMENTAL", "DIRECT": "DIRECT"}
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_2)}
            if evaluation == 'hard':
                entry["value"] = max(relevant, key=relevant.get)
            else:
                entry["value"] = relevant

        elif task == '2_3':
            label_map = {"NONE": "NO"}
            label_map.update({lbl: lbl for lbl in LABELS_2_3[1:]})
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_3)}

            if evaluation == 'hard':
                # Use per-class thresholds if provided, else 0.5
                threshs = thresholds_c if thresholds_c is not None else [0.5] * len(LABELS_2_3)
                # Map thresholds to the remapped label names
                thresh_map = {label_map[lbl]: threshs[idx] for idx, lbl in enumerate(LABELS_2_3)}
                hard_labels = [lbl for lbl, p in relevant.items() if p >= thresh_map[lbl]]
                if not hard_labels:
                    hard_labels = [max(relevant, key=relevant.get)]
                entry["value"] = hard_labels
            else:
                entry["value"] = relevant

        out.append(entry)
    return out

In [ ]:
TASK_2_2_HIERARCHY = {
    "YES": ["DIRECT", "JUDGEMENTAL"],
    "NO": []
}
TASK_2_3_HIERARCHY = {
    "YES": [
        "IDEOLOGICAL-INEQUALITY",
        "STEREOTYPING-DOMINANCE",
        "OBJECTIFICATION",
        "SEXUAL-VIOLENCE",
        "MISOGYNY-NON-SEXUAL-VIOLENCE"
    ],
    "NO": []
}

In [ ]:
def _extract_metrics_from_report(report, metric_list):
    """Pull metric scores out of a PyEvALL report object."""
    results = {}
    if not hasattr(report, 'report') or 'metrics' not in report.report:
        return results
    
    metrics_dict = report.report['metrics']
    for metric in metric_list:
        if metric not in metrics_dict:
            continue
        if metrics_dict[metric].get('status') != 'OK':
            continue
        try:
            results[metric] = float(
                metrics_dict[metric]['results']['average_per_test_case']
            )
        except (KeyError, TypeError, ValueError):
            pass
    return results


In [ ]:
def _run_pyevall(probs, gold_dist, sample_ids, task, evaluation,
                 temp_dir='./pyevall_temp_train', thresholds_c=None):
    """Single PyEvALL run for one task and evaluation mode.
    
    thresholds_c: per-class thresholds for Task C hard mode (list of 6 floats).
                  Ignored for all other tasks and for soft evaluation.
    """
    os.makedirs(temp_dir, exist_ok=True)

    preds = _format_for_pyevall(probs, sample_ids, task, evaluation,
                                 thresholds_c=thresholds_c)
    gold  = _format_for_pyevall(gold_dist, sample_ids, task, evaluation)
    # Note: thresholds_c is intentionally NOT passed for gold — gold uses raw
    # soft distributions and PyEvALL binarizes them internally for hard mode.

    pred_path = f'{temp_dir}/pred_{task}_{evaluation}.json'
    gold_path = f'{temp_dir}/gold_{task}_{evaluation}.json'
    with open(pred_path, 'w') as f: json.dump(preds, f)
    with open(gold_path, 'w') as f: json.dump(gold, f)

    if evaluation == 'hard':
        metric_list = ['ICM', 'ICMNorm', 'FMeasure']
    else:
        if task == '2_3':
            metric_list = ['ICMSoft', 'ICMSoftNorm']
        else:
            metric_list = ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']

    params = {}
    if task == '2_2':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_2_HIERARCHY
    elif task == '2_3':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_3_HIERARCHY

    try:
        evaluator = PyEvALLEvaluation()
        report    = evaluator.evaluate(pred_path, gold_path, metric_list, **params)
        return _extract_metrics_from_report(report, metric_list)
    except Exception as e:
        print(f"  ! PyEvALL failed for task {task} ({evaluation}): {e}")
        return {}


In [ ]:
def compute_metrics(eval_pred):
    """Compute all metrics using official PyEvALL library.

    Runs 6 evaluations (3 tasks × {hard, soft}) plus a per-class threshold
    search for Task C hard (72 extra PyEvALL calls via greedy coordinate search).
    """
    logits_a, logits_b, logits_c = eval_pred.predictions
    labels_a, labels_b, labels_c = eval_pred.label_ids
    logits_a = np.asarray(logits_a)
    logits_b = np.asarray(logits_b)
    logits_c = np.asarray(logits_c)
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    labels_c = np.asarray(labels_c)
    metrics = {}

    with tempfile.TemporaryDirectory() as temp_dir:

        # === Task A ===
        if labels_a.ndim == 2:
            labels_a = labels_a[:, 1]

        mask_a = labels_a != -100.0
        if mask_a.any():
            prob_yes  = 1 / (1 + np.exp(-logits_a[mask_a].reshape(-1)))
            probs_a  = np.stack([1 - prob_yes, prob_yes], axis=-1)
            targets_a = np.stack([1 - labels_a[mask_a], labels_a[mask_a]], axis=-1)
            ids_a    = [str(i) for i in range(len(probs_a))]

            hard_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'hard',
                                       temp_dir=temp_dir)
            for k, v in hard_scores.items():
                metrics[f'task_a_{k.lower()}_hard'] = v

            soft_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_a_{k.lower()}_soft'] = v

        # === Task B ===
        mask_b = labels_b[:, 0] != -1.0
        if mask_b.any():
            x        = logits_b[mask_b]
            x        = x - x.max(axis=-1, keepdims=True)
            probs_b  = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
            targets_b = labels_b[mask_b]
            ids_b    = [str(i) for i in range(len(probs_b))]

            hard_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'hard',
                                       temp_dir=temp_dir)
            for k, v in hard_scores.items():
                metrics[f'task_b_{k.lower()}_hard'] = v

            soft_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_b_{k.lower()}_soft'] = v

        # === Task C ===
        mask_c = labels_c[:, 0] != -1.0
        if mask_c.any():
            probs_c  = 1 / (1 + np.exp(-logits_c[mask_c]))
            targets_c = labels_c[mask_c]
            ids_c    = [str(i) for i in range(len(probs_c))]

            # Soft (unchanged)
            soft_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_c_{k.lower()}_soft'] = v

            # Hard — default threshold (0.5) for reference
            hard_scores_default = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                               temp_dir=temp_dir)
            for k, v in hard_scores_default.items():
                metrics[f'task_c_{k.lower()}_hard_default'] = v

            # Hard — tuned thresholds via greedy coordinate search
            best_thresholds, hard_scores_tuned = _search_thresholds_task_c(
                probs_c, targets_c, ids_c, temp_dir
            )
            for k, v in hard_scores_tuned.items():
                metrics[f'task_c_{k.lower()}_hard'] = v  # this becomes the canonical hard score

            # Log per-class thresholds so you can watch them evolve
            for j, label in enumerate(LABELS_2_3):
                metrics[f'task_c_thresh_{label.lower().replace("-", "_")}'] = round(best_thresholds[j], 2)

    return metrics

In [ ]:
import numpy as np
cvpc_values = [train_dataset[i]["cvpc"].item() for i in range(len(train_dataset))]
print(f"CVPC mean = {np.mean(cvpc_values):.4f}")
print(f"CVPC std  = {np.std(cvpc_values):.4f}")
print(f"CVPC range = [{min(cvpc_values):.4f}, {max(cvpc_values):.4f}]")

In [ ]:
# Add this once, anywhere after train_dataset is constructed:
sample_frames = train_dataset[0]["frames"]
assert sample_frames.shape == (16, 3, 224, 224), f"Got {sample_frames.shape}"
print(f"✓ Frame shape matches: {sample_frames.shape}")

In [ ]:
print([n for n, _ in model.named_parameters() if "tau" in n.lower()])

In [ ]:
optimizer = get_optimizer(model)
swa_callback = SWACallback(swa_start_epoch=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
    callbacks=[
        swa_callback,
    ],
)

trainer.label_names = ["labels_a", "labels_b", "labels_c"]

In [ ]:
trainer.train()

In [ ]:
final_metrics = trainer.evaluate()
print("[SWA] Final SWA model metrics:")
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

In [ ]:
import importlib
import ablation_harness
importlib.reload(ablation_harness)
from ablation_harness import run_all_ablations, print_ablation_table, save_for_paper

model = trainer.model
results = run_all_ablations(model, val_dataset, compute_metrics, device="cuda")
df = print_ablation_table(results)
df.to_csv(f"ablation_seed_{SEED}.csv", index=False)
save_for_paper(df)